# 03_train_3d — Launcher Colab (A100)

Unico punto di accesso per eseguire il training 3D BraTS-PEDs su Google Colab.
Segue esattamente `step.md` (Fase 2): monta Drive, clona il codice da GitHub,
porta i dati e i pesi pre-addestrati in locale su `/content/`, poi lancia
`run_pipeline_3d.py`.

**Prerequisiti (Fase 1, gia' completata):** i file `train_3d.zip`, `val_3d.zip`,
`test_3d.zip` e `split_3d.json` sono gia' presenti su Google Drive in
`MyDrive/BraTS_Project/data/`, e i pesi pre-addestrati (`model_swinvit.pt`,
`model_best_fold_0.pth` per SwinUNETR; `model.pt` per SegResNet — bundle MONAI
Model Zoo `brats_mri_segmentation`, `init_filters=16`) sono in
`MyDrive/BraTS_Project/pretrained/`. Il progetto e' 100% transfer learning: per
allenare SegResNet serve `model.pt` caricato su Drive in questa cartella prima
di eseguire il notebook.

**Runtime:** verificare `Runtime > Change runtime type > A100 GPU` prima di
eseguire la prima cella.

## 1. Mount Google Drive

Necessario per accedere ai dati (zip) e ai pesi pre-addestrati caricati
manualmente nella Fase 1. Autorizzare l'accesso quando richiesto dal popup.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 2. Setup Ambiente & Clone

Installa le dipendenze minime (`monai`, `nibabel`, `tqdm` — coerenti con
`requirements.txt`; `torch` e' gia' preinstallato con build CUDA nell'immagine
Colab standard, quindi non va reinstallato) e clona **esclusivamente** il
branch `estensione-pipeline-3d` del repository ufficiale, che contiene tutto
il codice sorgente (`src/`, `run_pipeline_3d.py`) gia' scritto e testato in
locale. Il clone NON porta dati ne' pesi: sono esclusi da `.gitignore` e
vengono recuperati da Drive nelle celle successive.

In [ ]:
!pip install -q monai nibabel tqdm

!git clone -b estensione-pipeline-3d https://github.com/Drastid/BraTS-PEDs-brain-tumour-segmentation.git BraTS-PEDs-3D

%cd BraTS-PEDs-3D

## 3. I/O — Dati

Crea `data/processed_3d/`, copia i tre archivi zip (train/val/test) e
`split_3d.json` da Drive allo storage locale della VM Colab (`/content/`,
molto piu' veloce di Drive montato via FUSE per le letture ripetute che MONAI
fara' durante il training), li decomprime silenziosamente e posiziona
`split_3d.json` nel punto atteso dal codice (`data/split_3d.json`).

In [ ]:
import os
import shutil

DRIVE_DATA = "/content/drive/MyDrive/BraTS_Project/data"

os.makedirs("data/processed_3d", exist_ok=True)

# Copia gli zip da Drive allo storage locale di Colab
for fname in ["train_3d.zip", "val_3d.zip", "test_3d.zip"]:
    shutil.copy(os.path.join(DRIVE_DATA, fname), os.path.join("data", fname))

# Decompressione silenziosa (-q) di ciascuno split dentro data/processed_3d/
!unzip -q data/train_3d.zip -d data/processed_3d/
!unzip -q data/val_3d.zip -d data/processed_3d/
!unzip -q data/test_3d.zip -d data/processed_3d/

# split_3d.json va copiato (non zippato) direttamente in data/
shutil.copy(os.path.join(DRIVE_DATA, "split_3d.json"), "data/split_3d.json")

print("train:", len(os.listdir("data/processed_3d/train")))
print("val  :", len(os.listdir("data/processed_3d/val")))
print("test :", len(os.listdir("data/processed_3d/test")))

## 4. Caricamento Pesi Pre-addestrati

**Il progetto 3D e' 100% transfer learning** (decisione utente): entrambe le
architetture (SegResNet, SwinUNETR) partono SEMPRE da pesi pre-addestrati, mai
da zero. I pesi NON vengono scaricati da internet a runtime: sono gia' stati
caricati manualmente su Drive (Fase 1) e vengono qui copiati fisicamente nella
cartella locale `weights/` del progetto Colab:

- `model_swinvit.pt` e/o `model_best_fold_0.pth` — checkpoint SwinUNETR (SSL
  NVIDIA o HuggingFace/BrainSegFounder).
- `model.pt` — checkpoint SegResNet (bundle MONAI Model Zoo
  `brats_mri_segmentation`, `init_filters=16`, verificato: 81/83 tensori del
  backbone caricano correttamente, solo la head 3→5 classi resta esclusa).

I flag pesi sono **specifici per architettura** (`--pretrained-swinunetr`,
`--pretrained-segresnet`): un checkpoint SwinUNETR non ha alcuna chiave in
comune con SegResNet, quindi non esiste un `--weights-path` condiviso. Con
`--pretrained=auto` (default), l'assenza del checkpoint per un'architettura
richiesta blocca l'esecuzione PRIMA del training (policy transfer learning
enforced in `run_pipeline_3d.py`).

In [ ]:
DRIVE_PRETRAINED = "/content/drive/MyDrive/BraTS_Project/pretrained"

os.makedirs("weights", exist_ok=True)

for fname in ["model_swinvit.pt", "model_best_fold_0.pth", "model.pt"]:
    shutil.copy(os.path.join(DRIVE_PRETRAINED, fname), os.path.join("weights", fname))

print(os.listdir("weights"))

## 5. Training

Lancia la pipeline di training via CLI (`run_pipeline_3d.py`). La barra di
progresso per epoca (tempo stimato, loss corrente) e' gestita **internamente**
dal training loop in `src/train_3d.py`, che avvolge gia' i batch del
`DataLoader` con `tqdm` (`train_one_epoch_3d`, `train_one_epoch_gsl_3d`,
`evaluate_3d`) — non serve alcuna configurazione aggiuntiva in questa cella
per vederla: appare automaticamente nell'output sotto.

Esempio: baseline DiceFocalLoss su SwinUNETR (`--arch swinunetr`) con i pesi
pre-addestrati HuggingFace/BrainSegFounder caricati dalla cella precedente
(`--pretrained-swinunetr`). Per allenare **entrambe** le architetture in
sequenza, usare la cella "5b" subito sotto (`--all`) invece di questa: pesca
automaticamente `--pretrained-segresnet`/`--pretrained-swinunetr` per l'arch
corrente del ciclo, e con `--pretrained=auto` entrambi sono obbligatori (nessun
training da zero per errore).

In [ ]:
!python run_pipeline_3d.py \
    --arch swinunetr \
    --loss dice_focal \
    --data-root data/processed_3d \
    --roi 128 128 128 \
    --batch-size 2 \
    --num-samples 2 \
    --num-workers 4 \
    --epochs 100 \
    --pretrained auto \
    --pretrained-swinunetr weights/model_best_fold_0.pth \
    --run-name run01 \
    --backup-dir /content/drive/MyDrive/BraTS_Project/checkpoints

### 5b. (Alternativa) Allenare ENTRAMBE le architetture in sequenza con `--all`

Sostituisce la cella precedente: allena SegResNet e poi SwinUNETR in sequenza,
ciascuna nella propria sottocartella `<run-name>/<arch>_<loss>/`. Con
`--pretrained=auto` (default) **entrambi** i flag pesi sono obbligatori — se ne
manca uno, lo script si ferma PRIMA di allenare qualsiasi modello (nessun
training da zero per errore, policy transfer learning enforced in
`_validate_pretrained_weights`). Un fallimento (es. OOM) su un'architettura non
cancella i risultati dell'altra già completata.

In [ ]:
!python run_pipeline_3d.py \
    --all \
    --loss dice_focal \
    --data-root data/processed_3d \
    --roi 128 128 128 \
    --batch-size 2 \
    --num-samples 2 \
    --num-workers 4 \
    --epochs 100 \
    --pretrained auto \
    --pretrained-segresnet weights/model.pt \
    --pretrained-swinunetr weights/model_best_fold_0.pth \
    --run-name run01 \
    --backup-dir /content/drive/MyDrive/BraTS_Project/checkpoints